# GT-DTW (Global Time Dynamic Time Warping)

### 2.1 Загрузка подготовленных данных

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import pickle
import gc
import warnings
import time
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
import os

warnings.filterwarnings('ignore')

with open('data/ts_prepared_data.pkl', 'rb') as f:
    ts_data = pickle.load(f)

clustering_patients = ts_data['clustering_patients']
clustering_series = ts_data['clustering_series']
clustering_dates = ts_data['clustering_dates']
clustering_kzs_count = ts_data['clustering_kzs_count']
patient_groups = ts_data['patient_groups']

print(f"   Загружено пациентов для кластеризации: {len(clustering_patients)}")
print(f"   Доступные показатели: {list(clustering_series.keys())}")

# Проверка длины рядов
lengths = [len(clustering_series['САД'][p]) for p in clustering_patients]
print(f"   Длина рядов: мин={min(lengths)}, макс={max(lengths)}, среднее={np.mean(lengths):.0f}")

   Загружено пациентов для кластеризации: 2500
   Доступные показатели: ['САД', 'ДАД', 'ЧП', 'пульсовое_давление']
   Длина рядов: мин=143, макс=279, среднее=199


### 2.2 Реализация GT-DTW

In [3]:
def gt_dtw_distance(series1, series2, dates1, dates2, 
                    window_days=7, gamma=1.0, metric='САД'):
    """
    GT-DTW расстояние между двумя временными рядами
    
    Parameters:
    -----------
    series1, series2 : array-like
        Значения показателей (нормализованные)
    dates1, dates2 : array-like
        Даты наблюдений (datetime64)
    window_days : int
        Максимальное отклонение во времени (дни)
    gamma : float
        Вес временного штрафа
    metric : str
        Метрика для сравнения ('САД', 'ДАД', 'ЧП', 'пульсовое_давление')
    
    Returns:
    --------
    float : GT-DTW расстояние
    """
    n, m = len(series1), len(series2)
    
    # Преобразуем datetime64 в дни от начала
    # Используем минимальную дату как reference point
    min_date = np.min([np.min(dates1), np.min(dates2)])
    
    # Вычисляем дни как (date - min_date) в днях
    t1 = np.array([(d - min_date) / np.timedelta64(1, 'D') for d in dates1])
    t2 = np.array([(d - min_date) / np.timedelta64(1, 'D') for d in dates2])
    
    # Инициализация матрицы DTW
    dtw_matrix = np.full((n + 1, m + 1), np.inf)
    dtw_matrix[0, 0] = 0
    
    # Заполняем матрицу
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            # Проверка временного окна
            time_diff = abs(t1[i-1] - t2[j-1])
            if time_diff > window_days:
                continue
            
            # Стоимость: разница значений + временной штраф
            value_cost = (series1[i-1] - series2[j-1]) ** 2
            time_penalty = gamma * (time_diff / window_days) ** 2
            
            cost = value_cost + time_penalty
            
            # Минимальная стоимость из трех переходов
            dtw_matrix[i, j] = cost + min(
                dtw_matrix[i-1, j],    # вставка
                dtw_matrix[i, j-1],    # удаление
                dtw_matrix[i-1, j-1]   # соответствие
            )
    
    return np.sqrt(dtw_matrix[n, m])

def gt_dtw_distance_fast(series1, series2, dates1, dates2, 
                         window_days=7, gamma=1.0, metric='САД'):
    """
    Быстрая версия GT-DTW с ограничением окна (Sakoe-Chiba band)
    """
    n, m = len(series1), len(series2)
    
    # Преобразуем datetime64 в дни
    min_date = np.min([np.min(dates1), np.min(dates2)])
    t1 = np.array([(d - min_date) / np.timedelta64(1, 'D') for d in dates1])
    t2 = np.array([(d - min_date) / np.timedelta64(1, 'D') for d in dates2])
    
    # Ограничиваем окно пропорционально длине рядов
    window_size = max(1, int(window_days / 7))  # приблизительное окно в шагах
    
    dtw_matrix = np.full((n + 1, m + 1), np.inf)
    dtw_matrix[0, 0] = 0
    
    for i in range(1, n + 1):
        # Ограничиваем j окном
        j_start = max(1, i - window_size)
        j_end = min(m, i + window_size)
        
        for j in range(j_start, j_end + 1):
            time_diff = abs(t1[i-1] - t2[j-1])
            if time_diff > window_days:
                continue
            
            value_cost = (series1[i-1] - series2[j-1]) ** 2
            time_penalty = gamma * (time_diff / window_days) ** 2
            
            cost = value_cost + time_penalty
            
            dtw_matrix[i, j] = cost + min(
                dtw_matrix[i-1, j],
                dtw_matrix[i, j-1],
                dtw_matrix[i-1, j-1]
            )
    
    return np.sqrt(dtw_matrix[n, m])

# Тестируем на первых двух пациентах
print("\nТестирование GT-DTW на первых двух пациентах:")
p1, p2 = clustering_patients[0], clustering_patients[1]

dist_sad = gt_dtw_distance(
    clustering_series['САД'][p1], clustering_series['САД'][p2],
    clustering_dates[p1], clustering_dates[p2],
    window_days=7, gamma=1.0
)
print(f"  Расстояние GT-DTW (САД): {dist_sad:.3f}")

# Тестируем с разными параметрами
print("\nВлияние параметров:")
for window in [3, 7, 14]:
    dist = gt_dtw_distance(
        clustering_series['САД'][p1], clustering_series['САД'][p2],
        clustering_dates[p1], clustering_dates[p2],
        window_days=window, gamma=1.0
    )
    print(f"  window={window}: {dist:.3f}")

for g in [0.0, 0.5, 1.0, 2.0]:
    dist = gt_dtw_distance(
        clustering_series['САД'][p1], clustering_series['САД'][p2],
        clustering_dates[p1], clustering_dates[p2],
        window_days=7, gamma=g
    )
    print(f"  gamma={g}: {dist:.3f}")


Тестирование GT-DTW на первых двух пациентах:
  Расстояние GT-DTW (САД): 15.906

Влияние параметров:
  window=3: 18.691
  window=7: 15.906
  window=14: 13.949
  gamma=0.0: 13.318
  gamma=0.5: 15.021
  gamma=1.0: 15.906
  gamma=2.0: 16.884


### 2.3 Выбор основного показателя и параметров

In [4]:
# Проверяем корреляцию между разными показателями
sample_size = min(50, len(clustering_patients))
sample_patients = clustering_patients[:sample_size]

correlations = {'САД-ДАД': [], 'САД-ЧП': [], 'ДАД-ЧП': []}

for i, p1 in enumerate(sample_patients):
    for p2 in sample_patients[i+1:]:
        if p1 in clustering_series['САД'] and p2 in clustering_series['САД']:
            # Расстояния по разным показателям
            d_sad = gt_dtw_distance(
                clustering_series['САД'][p1], clustering_series['САД'][p2],
                clustering_dates[p1], clustering_dates[p2],
                window_days=7, gamma=1.0
            )
            d_dad = gt_dtw_distance(
                clustering_series['ДАД'][p1], clustering_series['ДАД'][p2],
                clustering_dates[p1], clustering_dates[p2],
                window_days=7, gamma=1.0
            )
            d_chp = gt_dtw_distance(
                clustering_series['ЧП'][p1], clustering_series['ЧП'][p2],
                clustering_dates[p1], clustering_dates[p2],
                window_days=7, gamma=1.0
            )
            
            correlations['САД-ДАД'].append((d_sad, d_dad))
            correlations['САД-ЧП'].append((d_sad, d_chp))
            correlations['ДАД-ЧП'].append((d_dad, d_chp))

# Вычисляем корреляции
for key, pairs in correlations.items():
    if len(pairs) > 1:
        x, y = zip(*pairs)
        corr = np.corrcoef(x, y)[0, 1]
        print(f"  Корреляция расстояний {key}: {corr:.3f}")

# Выбираем САД как основной показатель (наиболее клинически значимый)
primary_metric = 'САД'
print(f"\nВыбран основной показатель: {primary_metric}")

  Корреляция расстояний САД-ДАД: 0.540
  Корреляция расстояний САД-ЧП: 0.166
  Корреляция расстояний ДАД-ЧП: 0.100

Выбран основной показатель: САД


### 2.4 Параллельное вычисление матрицы расстояний

In [11]:
import numpy as np
import time
from itertools import combinations
import pandas as pd

# Попытка импорта numba (опционально)
try:
    from numba import jit, prange
    HAS_NUMBA = True
except ImportError:
    HAS_NUMBA = False
    print("  numba не установлена, используем обычную версию")

# =============================================================================
# ОПТИМИЗИРОВАННАЯ ВЕРСИЯ С ИСПРАВЛЕНИЕМ ОШИБКИ ТИПОВ ДАТ
# =============================================================================

def prepare_time_data(dates_dict):
    """
    Подготовка временных меток: конвертация timedelta64 в дни
    """
    # Находим минимальную дату среди всех
    all_dates = []
    for d in dates_dict.values():
        if d is not None and len(d) > 0:
            all_dates.extend(d)
    
    if not all_dates:
        return {}
    
    # Конвертируем все даты в pandas Timestamp для удобства
    min_date = pd.Timestamp(min(all_dates))
    
    # Создаем словарь с днями от начала
    time_days = {}
    for patient, dates in dates_dict.items():
        if dates is not None and len(dates) > 0:
            # Конвертируем в pandas Timestamp если нужно
            if isinstance(dates[0], np.datetime64):
                dates_ts = [pd.Timestamp(d) for d in dates]
            else:
                dates_ts = dates
            
            # Вычисляем дни от начала
            days = np.array([(d - min_date).days for d in dates_ts], dtype=np.float32)
            time_days[patient] = days
    
    return time_days, min_date


@jit(nopython=True, cache=True)
def gt_dtw_distance_numba(s1, s2, t1, t2, window_days, gamma):
    """
    GT-DTW с JIT-компиляцией для максимальной скорости
    """
    n, m = len(s1), len(s2)
    
    # Ограничиваем окно пропорционально длине
    window_size = max(1, int(window_days / 7))
    
    # Инициализация матрицы
    dtw_matrix = np.full((n + 1, m + 1), np.inf)
    dtw_matrix[0, 0] = 0
    
    for i in range(1, n + 1):
        # Ограничиваем j окном
        j_start = max(1, i - window_size)
        j_end = min(m, i + window_size)
        
        for j in range(j_start, j_end + 1):
            time_diff = abs(t1[i-1] - t2[j-1])
            if time_diff > window_days:
                continue
            
            value_cost = (s1[i-1] - s2[j-1]) ** 2
            time_penalty = gamma * (time_diff / window_days) ** 2
            cost = value_cost + time_penalty
            
            dtw_matrix[i, j] = cost + min(
                dtw_matrix[i-1, j],
                dtw_matrix[i, j-1],
                dtw_matrix[i-1, j-1]
            )
    
    return np.sqrt(dtw_matrix[n, m])


def gt_dtw_distance_fast(s1, s2, t1, t2, window_days, gamma):
    """
    Быстрая версия без numba (если numba не доступна)
    """
    n, m = len(s1), len(s2)
    window_size = max(1, int(window_days / 7))
    
    dtw_matrix = np.full((n + 1, m + 1), np.inf)
    dtw_matrix[0, 0] = 0
    
    for i in range(1, n + 1):
        j_start = max(1, i - window_size)
        j_end = min(m, i + window_size)
        
        for j in range(j_start, j_end + 1):
            time_diff = abs(t1[i-1] - t2[j-1])
            if time_diff > window_days:
                continue
            
            value_cost = (s1[i-1] - s2[j-1]) ** 2
            time_penalty = gamma * (time_diff / window_days) ** 2
            cost = value_cost + time_penalty
            
            dtw_matrix[i, j] = cost + min(
                dtw_matrix[i-1, j],
                dtw_matrix[i, j-1],
                dtw_matrix[i-1, j-1]
            )
    
    return np.sqrt(dtw_matrix[n, m])


def compute_distance_matrix_optimized(patients, series_dict, dates_dict, 
                                      metric='САД', window_days=7, gamma=1.0):
    """
    ОПТИМИЗИРОВАННАЯ версия с исправленной обработкой дат
    """
    n = len(patients)
    
    print("  Подготовка временных меток...")
    # Подготавливаем временные метки (дни от начала)
    time_days, min_date = prepare_time_data(dates_dict)
    print(f"  Минимальная дата: {min_date}")
    
    print("  Подготовка данных пациентов...")
    # Создаем списки всех рядов и временных меток
    all_series = []
    all_times = []
    valid_patients = []
    valid_indices = []
    
    for idx, p in enumerate(patients):
        s = series_dict[metric].get(p)
        t = time_days.get(p)
        
        if s is not None and t is not None and len(s) > 0 and len(t) > 0:
            all_series.append(s.astype(np.float32))
            all_times.append(t)
            valid_patients.append(p)
            valid_indices.append(idx)
    
    n_valid = len(valid_patients)
    print(f"  Валидных пациентов: {n_valid}/{n}")
    
    if n_valid == 0:
        print("  Нет валидных пациентов!")
        return np.array([]), []
    
    # Инициализируем матрицу расстояний
    distance_matrix = np.zeros((n_valid, n_valid), dtype=np.float32)
    
    total_pairs = n_valid * (n_valid - 1) // 2
    print(f"  Всего пар для вычисления: {total_pairs:,}")
    
    # Выбираем функцию расстояния
    if HAS_NUMBA:
        dist_func = gt_dtw_distance_numba
        print("  Используем JIT-компилированную версию")
    else:
        dist_func = gt_dtw_distance_fast
        print("  Используем обычную версию")
    
    # Вычисляем расстояния
    start_time = time.time()
    processed = 0
    
    # Используем combinations для эффективного перебора
    for i, j in combinations(range(n_valid), 2):
        dist = dist_func(
            all_series[i], all_series[j],
            all_times[i], all_times[j],
            window_days, gamma
        )
        
        distance_matrix[i, j] = dist
        distance_matrix[j, i] = dist
        
        processed += 1
        
        # Прогресс каждые 5%
        if processed % max(1, total_pairs // 20) == 0:
            elapsed = time.time() - start_time
            pct = processed / total_pairs * 100
            if processed > 0:
                est_total = elapsed / (processed / total_pairs)
                remaining = est_total - elapsed
                print(f"    Прогресс: {pct:.1f}% ({processed}/{total_pairs}), "
                      f"прошло: {elapsed:.1f}с, осталось: {remaining:.1f}с")
            else:
                print(f"    Прогресс: {pct:.1f}% ({processed}/{total_pairs})")
    
    total_time = time.time() - start_time
    print(f"  Матрица вычислена за {total_time:.1f} сек")
    
    # Оценка скорости
    if processed > 0:
        avg_time = total_time / processed
        print(f"  Среднее время на пару: {avg_time*1000:.2f} мс")
    
    return distance_matrix, valid_patients


# =============================================================================
# ВЕРСИЯ С БАТЧАМИ ДЛЯ ЭКОНОМИИ ПАМЯТИ
# =============================================================================

def compute_distance_matrix_batched(patients, series_dict, dates_dict,
                                    metric='САД', window_days=7, gamma=1.0,
                                    batch_size=500):
    """
    Версия с батчами для очень больших данных (экономия памяти)
    """
    n = len(patients)
    
    print("  Подготовка временных меток...")
    time_days, min_date = prepare_time_data(dates_dict)
    
    print("  Подготовка данных...")
    # Создаем списки данных
    valid_data = []
    valid_patients = []
    
    for p in patients:
        s = series_dict[metric].get(p)
        t = time_days.get(p)
        
        if s is not None and t is not None and len(s) > 0 and len(t) > 0:
            valid_data.append({
                'patient': p,
                'series': s.astype(np.float32),
                'times': t
            })
            valid_patients.append(p)
    
    n_valid = len(valid_patients)
    print(f"  Валидных пациентов: {n_valid}/{n}")
    
    if n_valid == 0:
        return np.array([]), []
    
    # Создаем матрицу расстояний
    distance_matrix = np.zeros((n_valid, n_valid), dtype=np.float32)
    
    # Разбиваем на батчи
    n_batches = (n_valid + batch_size - 1) // batch_size
    print(f"  Разбито на {n_batches} батчей по {batch_size} пациентов")
    
    start_time = time.time()
    total_pairs = n_valid * (n_valid - 1) // 2
    processed = 0
    
    for b1 in range(n_batches):
        start1 = b1 * batch_size
        end1 = min((b1 + 1) * batch_size, n_valid)
        
        for b2 in range(b1, n_batches):
            start2 = b2 * batch_size
            end2 = min((b2 + 1) * batch_size, n_valid)
            
            # Вычисляем расстояния между батчами
            for i in range(start1, end1):
                for j in range(max(i+1, start2), end2):
                    if i == j:
                        continue
                        
                    dist = gt_dtw_distance_fast(
                        valid_data[i]['series'], valid_data[j]['series'],
                        valid_data[i]['times'], valid_data[j]['times'],
                        window_days, gamma
                    )
                    
                    distance_matrix[i, j] = dist
                    distance_matrix[j, i] = dist
                    
                    processed += 1
                    
                    if processed % 10000 == 0:
                        elapsed = time.time() - start_time
                        pct = processed / total_pairs * 100
                        print(f"    Прогресс: {pct:.1f}% ({processed}/{total_pairs}), "
                              f"прошло: {elapsed:.1f}с")
    
    total_time = time.time() - start_time
    print(f"  Матрица вычислена за {total_time:.1f} сек")
    
    return distance_matrix, valid_patients


# =============================================================================
# ИСПОЛЬЗОВАНИЕ
# =============================================================================

print(f"\nВычисление матрицы расстояний для {len(clustering_patients)} пациентов...")

# Выбираем стратегию в зависимости от размера данных
n_patients = len(clustering_patients)

if n_patients > 2000:
    print(f"  Много пациентов ({n_patients}), используем батчевую версию")
    distance_matrix, valid_patients = compute_distance_matrix_batched(
        clustering_patients,
        clustering_series,
        clustering_dates,
        metric='САД',
        window_days=7,
        gamma=1.0,
        batch_size=300  # меньший размер батча для экономии памяти
    )
else:
    print(f"  Используем оптимизированную версию")
    distance_matrix, valid_patients = compute_distance_matrix_optimized(
        clustering_patients,
        clustering_series,
        clustering_dates,
        metric='САД',
        window_days=7,
        gamma=1.0
    )

# Обновляем список пациентов
clustering_patients = valid_patients

# Проверка матрицы
if len(distance_matrix) > 0:
    print(f"\nПроверка матрицы расстояний:")
    print(f"  Форма: {distance_matrix.shape}")
    print(f"  Тип данных: {distance_matrix.dtype}")
    print(f"  Использовано памяти: {distance_matrix.nbytes / 1024**2:.1f} MB")
    
    # Статистика (только верхний треугольник)
    triu_indices = np.triu_indices_from(distance_matrix, k=1)
    distances = distance_matrix[triu_indices]
    
    if len(distances) > 0:
        print(f"  Минимальное расстояние: {np.min(distances):.3f}")
        print(f"  Максимальное расстояние: {np.max(distances):.3f}")
        print(f"  Среднее расстояние: {np.mean(distances):.3f}")
        print(f"  Медиана расстояния: {np.median(distances):.3f}")
    
    # Сохраняем матрицу
    np.save('data/gt_dtw_distance_matrix.npy', distance_matrix)
    print(f"\nСохранено: data/gt_dtw_distance_matrix.npy")
    
    # Сохраняем обновленный список пациентов
    import pickle
    with open('data/clustering_patients_updated.pkl', 'wb') as f:
        pickle.dump(clustering_patients, f)
    print(f"Сохранено: data/clustering_patients_updated.pkl")
else:
    print("  Ошибка: матрица расстояний пуста!")


Вычисление матрицы расстояний для 2500 пациентов...
  Много пациентов (2500), используем батчевую версию
  Подготовка временных меток...
  Подготовка данных...
  Валидных пациентов: 2500/2500
  Разбито на 9 батчей по 300 пациентов
    Прогресс: 0.3% (10000/3123750), прошло: 9.9с
    Прогресс: 0.6% (20000/3123750), прошло: 19.5с
    Прогресс: 1.0% (30000/3123750), прошло: 29.3с
    Прогресс: 1.3% (40000/3123750), прошло: 39.1с
    Прогресс: 1.6% (50000/3123750), прошло: 46.4с
    Прогресс: 1.9% (60000/3123750), прошло: 52.2с
    Прогресс: 2.2% (70000/3123750), прошло: 58.1с
    Прогресс: 2.6% (80000/3123750), прошло: 64.1с
    Прогресс: 2.9% (90000/3123750), прошло: 70.2с
    Прогресс: 3.2% (100000/3123750), прошло: 76.4с
    Прогресс: 3.5% (110000/3123750), прошло: 82.5с
    Прогресс: 3.8% (120000/3123750), прошло: 88.5с
    Прогресс: 4.2% (130000/3123750), прошло: 94.8с
    Прогресс: 4.5% (140000/3123750), прошло: 100.1с
    Прогресс: 4.8% (150000/3123750), прошло: 104.6с
    Прогрес

Какой кеш создает этот код? даже например haggingfase или torch а в этой части кода? что можно удалить? Если с использованием кэширования, векторизации и многопоточности то нужно удалить весь созданный кеш все переменные временные данные

### 2.5 Визуализация матрицы расстояний

In [7]:
# Сортируем пациентов по количеству КЗС для лучшей визуализации
sorted_indices = np.argsort([clustering_kzs_count.get(p, 0) for p in clustering_patients])
sorted_matrix = distance_matrix[sorted_indices][:, sorted_indices]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Тепловая карта матрицы расстояний
im1 = axes[0].imshow(sorted_matrix, cmap='viridis', aspect='auto')
axes[0].set_title('Матрица GT-DTW расстояний\n(отсортировано по частоте КЗС)')
axes[0].set_xlabel('Пациенты')
axes[0].set_ylabel('Пациенты')
plt.colorbar(im1, ax=axes[0], label='Расстояние')

# 2. Распределение расстояний
axes[1].hist(distance_matrix[distance_matrix > 0].flatten(), bins=50, 
             edgecolor='black', alpha=0.7)
axes[1].axvline(np.median(distance_matrix[distance_matrix > 0]), 
                color='red', linestyle='--', label=f'Медиана: {np.median(distance_matrix):.2f}')
axes[1].set_xlabel('GT-DTW расстояние')
axes[1].set_ylabel('Частота')
axes[1].set_title('Распределение расстояний')
axes[1].legend()

# 3. Связь расстояния с разницей в КЗС
kzs_diff_matrix = np.zeros_like(distance_matrix)
for i in range(len(clustering_patients)):
    for j in range(len(clustering_patients)):
        kzs_i = clustering_kzs_count.get(clustering_patients[i], 0)
        kzs_j = clustering_kzs_count.get(clustering_patients[j], 0)
        kzs_diff_matrix[i, j] = abs(kzs_i - kzs_j)

# Сэмплируем для scatter plot
sample_size = min(10000, distance_matrix.size)
flat_dist = distance_matrix.flatten()
flat_kzs_diff = kzs_diff_matrix.flatten()
sample_idx = np.random.choice(len(flat_dist), sample_size, replace=False)

axes[2].scatter(flat_kzs_diff[sample_idx], flat_dist[sample_idx], 
                alpha=0.1, s=1)
axes[2].set_xlabel('Разница в количестве КЗС')
axes[2].set_ylabel('GT-DTW расстояние')
axes[2].set_title('Связь расстояния и разницы в КЗС')

# Добавляем линию тренда
z = np.polyfit(flat_kzs_diff[sample_idx], flat_dist[sample_idx], 1)
p = np.poly1d(z)
x_line = np.linspace(flat_kzs_diff.min(), flat_kzs_diff.max(), 100)
axes[2].plot(x_line, p(x_line), "r--", alpha=0.8, 
             label=f'Тренд: y={z[0]:.3f}x+{z[1]:.2f}')
axes[2].legend()

plt.tight_layout()
plt.savefig('data/photo/gt_dtw_distance_matrix.png', dpi=150)
plt.show()
print("Сохранено: data/photo/gt_dtw_distance_matrix.png")

NameError: name 'distance_matrix' is not defined

In [ ]:


# =============================================================================
# 2.6 Агломеративная кластеризация
# =============================================================================
print("\n" + "=" * 70)
print("2.6 Агломеративная кластеризация")
print("=" * 70)

from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform

# Преобразуем в форму для linkage
condensed_dist = squareform(distance_matrix)

# Пробуем разные методы linkage
print("Вычисление linkage матриц...")
methods = ['ward', 'complete', 'average']
linkage_matrices = {}

for method in methods:
    print(f"  Метод {method}...")
    linkage_matrices[method] = linkage(condensed_dist, method=method)

# Визуализируем дендрограммы
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (method, link_mat) in zip(axes, linkage_matrices.items()):
    # Усеченная дендрограмма для наглядности
    dendrogram(link_mat, ax=ax, truncate_mode='level', p=6,
               show_leaf_counts=True, leaf_rotation=90.)
    ax.set_title(f'Дендрограмма (метод: {method})')
    ax.set_xlabel('Пациенты')
    ax.set_ylabel('Расстояние')

plt.tight_layout()
plt.savefig('data/photo/dendrograms_comparison.png', dpi=150)
plt.show()
print("Сохранено: data/photo/dendrograms_comparison.png")

# =============================================================================
# 2.7 Определение оптимального числа кластеров
# =============================================================================
print("\n" + "=" * 70)
print("2.7 Определение оптимального числа кластеров")
print("=" * 70)

from sklearn.metrics import silhouette_score

# Используем метод Ward (обычно лучший)
linkage_matrix = linkage_matrices['ward']

# Анализ silhouette score для разного числа кластеров
k_range = range(2, min(16, len(clustering_patients) // 10))
silhouette_scores = []
inertia = []

print("Анализ silhouette score...")
for k in k_range:
    labels = fcluster(linkage_matrix, k, criterion='maxclust')
    score = silhouette_score(distance_matrix, labels, metric='precomputed')
    silhouette_scores.append(score)
    
    # Вычисляем "инерцию" (сумму квадратов внутри кластеров)
    cluster_inertia = 0
    for cluster_id in range(1, k + 1):
        cluster_points = np.where(labels == cluster_id)[0]
        if len(cluster_points) > 1:
            cluster_distances = distance_matrix[np.ix_(cluster_points, cluster_points)]
            cluster_inertia += np.sum(cluster_distances) / 2
    inertia.append(cluster_inertia)
    
    print(f"  k={k}: silhouette={score:.3f}")

# Находим оптимальное k
optimal_k_silhouette = k_range[np.argmax(silhouette_scores)]
print(f"\nОптимальное k по silhouette: {optimal_k_silhouette}")

# Метод локтя
k_derivative = np.diff(inertia)
k_elbow = k_range[np.argmin(k_derivative) + 1] if len(k_derivative) > 0 else 3
print(f"Оптимальное k по методу локтя: {k_elbow}")

# Выбираем k
optimal_k = min(optimal_k_silhouette, k_elbow)  # или можно взять среднее
print(f"Выбрано число кластеров: {optimal_k}")

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Silhouette scores
axes[0].plot(k_range, silhouette_scores, 'bo-')
axes[0].axvline(x=optimal_k_silhouette, color='red', linestyle='--', 
                label=f'Оптимум: {optimal_k_silhouette}')
axes[0].set_xlabel('Число кластеров (k)')
axes[0].set_ylabel('Silhouette Score')
axes[0].set_title('Silhouette анализ')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Метод локтя
axes[1].plot(k_range, inertia, 'ro-')
axes[1].axvline(x=k_elbow, color='blue', linestyle='--', 
                label=f'Локоть: {k_elbow}')
axes[1].set_xlabel('Число кластеров (k)')
axes[1].set_ylabel('Инерция (сумма расстояний внутри кластеров)')
axes[1].set_title('Метод локтя')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/photo/optimal_k_analysis.png', dpi=150)
plt.show()
print("Сохранено: data/photo/optimal_k_analysis.png")

# =============================================================================
# 2.8 Получение финальных кластеров
# =============================================================================
print("\n" + "=" * 70)
print("2.8 Получение финальных кластеров")
print("=" * 70)

# Получаем метки кластеров
cluster_labels = fcluster(linkage_matrix, optimal_k, criterion='maxclust')

# Создаем датафрейм с информацией о кластерах
cluster_df = pd.DataFrame({
    'patient_id': clustering_patients,
    'cluster': cluster_labels,
    'kzs_count': [clustering_kzs_count.get(p, 0) for p in clustering_patients],
    'group': [patient_groups.get(p, 'unknown') for p in clustering_patients]
})

# Статистика по кластерам
print(f"\nРаспределение по кластерам:")
cluster_stats = cluster_df.groupby('cluster').agg({
    'patient_id': 'count',
    'kzs_count': ['mean', 'median', 'std']
}).round(1)
cluster_stats.columns = ['size', 'kzs_mean', 'kzs_median', 'kzs_std']
print(cluster_stats)

print(f"\nРаспределение групп наблюдения по кластерам:")
group_by_cluster = pd.crosstab(
    cluster_df['cluster'], 
    cluster_df['group'],
    normalize='index'
) * 100
print(group_by_cluster.round(1))

# Сохраняем кластеры
cluster_df.to_csv('data/clustering_results.csv', index=False)
print(f"\nСохранено: data/clustering_results.csv")

# =============================================================================
# 2.9 Визуализация кластеров
# =============================================================================
print("\n" + "=" * 70)
print("2.9 Визуализация кластеров")
print("=" * 70)

# Функция для вычисления DTW центроида
def dtw_barycenter_averaging(series_list, max_iter=10):
    """Упрощенная версия DBA"""
    if not series_list:
        return None
    
    # Используем самую длинную серию как начальное приближение
    longest_idx = np.argmax([len(s) for s in series_list])
    centroid = series_list[longest_idx].copy()
    
    # Несколько итераций усреднения
    for _ in range(max_iter):
        aligned_sum = np.zeros_like(centroid, dtype=float)
        aligned_count = np.zeros_like(centroid, dtype=int)
        
        for series in series_list:
            # Простое выравнивание по длине (для демонстрации)
            if len(series) >= len(centroid):
                aligned = series[:len(centroid)]
            else:
                # Интерполяция
                x_old = np.linspace(0, 1, len(series))
                x_new = np.linspace(0, 1, len(centroid))
                aligned = np.interp(x_new, x_old, series)
            
            aligned_sum += aligned
            aligned_count += 1
        
        new_centroid = aligned_sum / aligned_count
        if np.allclose(centroid, new_centroid):
            break
        centroid = new_centroid
    
    return centroid

# Визуализируем центроиды кластеров
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for cluster_id in range(1, optimal_k + 1):
    if cluster_id > len(axes):
        break
        
    ax = axes[cluster_id - 1]
    
    cluster_patients = cluster_df[cluster_df['cluster'] == cluster_id]['patient_id'].tolist()
    
    # Получаем ряды для этих пациентов
    cluster_series = []
    for p in cluster_patients[:10]:  # ограничиваем для скорости
        if p in clustering_series['САД']:
            cluster_series.append(clustering_series['САД'][p])
    
    if cluster_series:
        # Отображаем несколько примеров
        for series in cluster_series[:5]:
            days = np.arange(len(series))
            ax.plot(days, series, alpha=0.3, linewidth=0.5, color='gray')
        
        # Вычисляем и отображаем центроид
        centroid = dtw_barycenter_averaging(cluster_series)
        if centroid is not None:
            days = np.arange(len(centroid))
            ax.plot(days, centroid, linewidth=2, color='red', label='Центроид')
        
        # Статистика
        kzs_mean = cluster_df[cluster_df['cluster'] == cluster_id]['kzs_count'].mean()
        size = len(cluster_patients)
        ax.set_title(f'Кластер {cluster_id} (n={size}, КЗС={kzs_mean:.1f})')
        ax.set_xlabel('Дни')
        ax.set_ylabel('САД (норм.)')
        ax.legend()
        ax.grid(True, alpha=0.3)

# Убираем лишние подграфики
for idx in range(optimal_k, len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.savefig('data/photo/cluster_centroids.png', dpi=150)
plt.show()
print("Сохранено: data/photo/cluster_centroids.png")

# =============================================================================
# 2.10 Сохранение всех результатов для следующих этапов
# =============================================================================
print("\n" + "=" * 70)
print("2.10 Сохранение результатов для следующих этапов")
print("=" * 70)

# Сохраняем все результаты
results = {
    'clustering_patients': clustering_patients,
    'cluster_labels': cluster_labels,
    'cluster_df': cluster_df,
    'distance_matrix': distance_matrix,
    'linkage_matrix': linkage_matrix,
    'optimal_k': optimal_k,
    'silhouette_scores': silhouette_scores,
    'k_range': list(k_range),
    'params': {
        'window_days': 7,
        'gamma': 1.0,
        'metric': 'САД'
    }
}

with open('data/clustering_results.pkl', 'wb') as f:
    pickle.dump(results, f)

print(f"Сохранено: data/clustering_results.pkl")

# =============================================================================
# ИТОГОВЫЙ ОТЧЕТ
# =============================================================================
print("\n" + "=" * 70)
print("ИТОГ ЭТАПА 2")
print("=" * 70)

print(f"""
ВЫПОЛНЕНО:

1. Реализован GT-DTW:
   - Функция с временным окном {results['params']['window_days']} дней
   - Параметр gamma = {results['params']['gamma']}
   - Основной показатель: {results['params']['metric']}

2. Вычислена матрица расстояний:
   - Пациентов: {len(clustering_patients)}
   - Размер матрицы: {distance_matrix.shape}
   - Диапазон расстояний: {np.min(distance_matrix[distance_matrix > 0]):.3f} - {np.max(distance_matrix):.3f}

3. Проведена агломеративная кластеризация:
   - Метод linkage: ward
   - Оптимальное число кластеров: {optimal_k}
   - Silhouette score: {np.max(silhouette_scores):.3f}

4. Характеристики кластеров:
""")

for cluster_id in range(1, optimal_k + 1):
    cluster_data = cluster_df[cluster_df['cluster'] == cluster_id]
    print(f"   Кластер {cluster_id}: {len(cluster_data)} пациентов, "
          f"ср. КЗС={cluster_data['kzs_count'].mean():.1f}")

print(f"""
СОХРАНЕННЫЕ ФАЙЛЫ:
- data/gt_dtw_distance_matrix.npy - матрица расстояний
- data/clustering_results.csv - метки кластеров для пациентов
- data/clustering_results.pkl - все результаты для следующих этапов
- data/photo/gt_dtw_distance_matrix.png - визуализация матрицы
- data/photo/dendrograms_comparison.png - дендрограммы
- data/photo/optimal_k_analysis.png - анализ оптимального k
- data/photo/cluster_centroids.png - центроиды кластеров

СЛЕДУЮЩИЙ ЭТАП (23 марта):
Фича-инжиниринг и вектора Шепли
""")

# Очистка памяти
#del distance_matrix, linkage_matrix, clustering_series, clustering_dates
del cluster_df, results
gc.collect()
print("\nПамять очищена")